In [1]:
import os
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import math

from utils.helpers import *
from implementations import ridge_regression
from utils.metrics import *

In [2]:
x_train, x_test, y_train, _, _ = load_csv_data(data_path='data/')

# 1. On preprocessed data 

In [3]:
data_path = 'data/preprocessed'

x_train_preprocessed = np.load(os.path.join(data_path, 'x_train_preprocessed.npy'))
x_test_preprocessed  = np.load(os.path.join(data_path, 'x_test_preprocessed.npy'))

print("Loaded shapes:")
print("x_train:", x_train_preprocessed.shape)
print("x_test: ", x_test_preprocessed.shape)

Loaded shapes:
x_train: (328135, 174)
x_test:  (109379, 174)


In [4]:
w, mse = ridge_regression(y_train, x_train_preprocessed, lambda_= 0.1)

pred_train = x_train_preprocessed @ w 

print("Scores ridge regression preprocessed data: ")
print(f"Accuracy:   {compute_accuracy(y_train, pred_train):.4f}")
print(f"Precision:  {compute_precision(y_train, pred_train):.4f}")
print(f"Recall:     {compute_recall(y_train, pred_train):.4f}")
print(f"F1 Score:   {compute_f1(y_train, pred_train):.4f}")
print(f"R2 Score:   {compute_r2(y_train, pred_train):.4f}")

Scores ridge regression preprocessed data: 
Accuracy:   0.6493
Precision:  0.1885
Recall:     0.8988
F1 Score:   0.3116
R2 Score:   -1.9431


In [5]:
print("1. NaNs in x_train_preprocessed:", np.isnan(x_train_preprocessed).any())
print("2. NaNs in y_train:             ", np.isnan(y_train).any())
print("3. NaNs in weight vector w:     ", np.isnan(w).any())

1. NaNs in x_train_preprocessed: False
2. NaNs in y_train:              False
3. NaNs in weight vector w:      False


# 2. Computing optimal lambda: 
## Cross validation

In [ ]:
from utils.costs import mse


def build_k_indices(y, k_fold, seed):
    """Build k indices for k-fold CV."""
    num_row = y.shape[0]
    interval = int(num_row / k_fold)
    np.random.seed(seed)
    indices = np.random.permutation(num_row)
    k_indices = [
        indices[k * interval : (k + 1) * interval] for k in range(k_fold)
    ]
    return np.array(k_indices)


def cross_validation(y, x, k_indices, k, lambda_):
    """Return the test loss of ridge regression for fold k."""
    # Split fold indices
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)].reshape(-1)

    y_te, y_tr = y[te_indice], y[tr_indice]
    x_te, x_tr = x[te_indice], x[tr_indice]

    # Fit Ridge Regression strictly on training fold
    w, _ = ridge_regression(y_tr, x_tr, lambda_)

    # Compute test RMSE on unseen test fold
    loss_te = np.sqrt(2 * mse(y_te - x_te @ w))

    return float(loss_te)


def cross_validation_demo(k_fold, lambdas):
    """Cross validation over regularization parameter lambda."""
    seed = 12
    k_indices = build_k_indices(y_train, k_fold, seed)

    rmse_te = []

    for lambda_ in lambdas:
        rmse_te_tmp = []
        for k in range(k_fold):
            loss_te = cross_validation(
                y_train, x_train_preprocessed, k_indices, k, lambda_
            )
            rmse_te_tmp.append(loss_te)
        rmse_te.append(np.mean(rmse_te_tmp))

    best_lambda = lambdas[np.argmin(rmse_te)]
    best_rmse = np.min(rmse_te)

    print(
        f"The choice of lambda which leads to the best test RMSE is {best_lambda:.5f} with a test RMSE of {best_rmse:.3f}"
    )
    return best_lambda, best_rmse


# Run cross-validation across lambda grid
best_lambda, best_rmse = cross_validation_demo(4, np.logspace(-4, 0, 30))

TypeError: mse() takes 1 positional argument but 3 were given

### Visualization

In [ ]:
#TODO